# Test the Quran reciter model

This notebook runs [ahmaher04/quran-reciter-detector](https://huggingface.co/ahmaher04/quran-reciter-detector).

One clip does two jobs:

1. **Reciter.** NVIDIA TitaNet-Small turns the audio into 192-d voiceprints. The notebook averages 20-second windows (hop 10 seconds) and names the first candidate, including when the cosine is under **0.70**.
2. **Surah and ayah.** [tarteel-ai/whisper-base-ar-quran](https://huggingface.co/tarteel-ai/whisper-base-ar-quran) writes an Arabic transcript. That text is aligned to all 6,236 ayahs.

A clip of **15 seconds or less is refused**. Pass **at least 20 seconds**. **30 seconds** is the length the model is scored at.

The last cell opens a page with **Upload** and **Microphone**. If your browser asks for microphone access, choose **Allow**.


## 1. Install

The first run downloads the voice encoder (~40 MB) and, on the first real clip, the Whisper checkpoint. Both are cached after that. After running the install cell, restart the notebook kernel before continuing.


In [7]:
%pip install -q "numpy>=1.24" "huggingface_hub>=0.26" "sherpa-onnx>=1.10" "soundfile>=0.12" "torch>=2.0" "transformers>=4.40" "gradio>=4.44"


Note: you may need to restart the kernel to use updated packages.


## 2. Load the gallery

Uses this folder when `catalog.json` is next to the notebook. Otherwise downloads the model repo from the Hub.


In [8]:
import sys
from pathlib import Path

import numpy as np
from huggingface_hub import snapshot_download

REPO = "ahmaher04/quran-reciter-detector"

here = Path.cwd()
if (here / "catalog.json").is_file() and (here / "reciter_identifier.py").is_file():
    root = here
else:
    root = Path(snapshot_download(REPO))

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from reciter_identifier import (
    LISTEN_SECONDS,
    MIN_SECONDS,
    ReciterIdentifier,
    download_encoder,
)
from verse_locator import VerseLocator, transcribe

gallery = ReciterIdentifier.from_pretrained(root)
encoder_path = download_encoder(root / "nemo_en_titanet_small.onnx", root / "config.json")
verses = VerseLocator(root / "quran")

print(f"root        {root}")
print(f"voices      {len(gallery.voices)}")
print(f"reciters    {gallery.reciters}")
print(f"dim         {gallery.dim}")
print(f"threshold   {gallery.threshold}")
print(f"listen      {LISTEN_SECONDS}s   minimum {MIN_SECONDS}s")
print(f"encoder     {encoder_path}")
print(f"ayah index  {len(verses._words)} words")


root        /Users/ahmedmaher/StudioProjects/muslim/huggingface/quran-reciter-id
voices      289
reciters    243
dim         192
threshold   0.7
listen      30s   minimum 20s
encoder     /Users/ahmedmaher/StudioProjects/muslim/huggingface/quran-reciter-id/nemo_en_titanet_small.onnx
ayah index  77794 words


## 3. Check the gallery

Row 0 must match itself at cosine 1. That only proves the file loaded. It is not a recording of a reciter.


In [9]:
matches, best = gallery.identify(gallery.embeddings[0])
voice = gallery.voices[0]
assert best is not None and best.voice.recording_id == voice.recording_id
assert abs(best.score - 1.0) < 1e-5
print(f"{voice.name_ar} / {voice.name_en}")
print(f"{voice.recording_name}")
print(f"self cosine {best.score:.6f}")
print("top 3 on that same vector:")
for match in matches:
    print(f"  {match.score:.3f}  {match.voice.name_en}  {match.voice.recording_name}")


عبدالباسط عبدالصمد / Abdulbasit Abdulsamad
المصحف المجود - المصحف المجود
self cosine 1.000000
top 3 on that same vector:
  1.000  Abdulbasit Abdulsamad  المصحف المجود - المصحف المجود
  0.721  Mohammed Siddiq Al-Minshawi  المصحف المعلم - المصحف المعلم
  0.684  Mohammed Siddiq Al-Minshawi  المصحف المجود - المصحف المجود


## 4. Surah and ayah from text

No audio yet. A line that exists in more than one ayah stays unknown. `الحمد لله رب العالمين` is 1:2, 10:10, 39:75, and 40:65, so the locator refuses it until the next words arrive.


In [10]:
samples = [
    "إياك نعبد وإياك نستعين اهدنا الصراط المستقيم",
    "قل هو الله أحد الله الصمد لم يلد ولم يولد",
    "الله لا إله إلا هو الحي القيوم لا تأخذه سنة ولا نوم",
    "الحمد لله رب العالمين",
    "فبأي آلاء ربكما تكذبان",
]

for text in samples:
    hit = verses.locate(text)
    if hit is None:
        print(f"unknown   {text}")
        continue
    span = f"{hit.ayah}" if (hit.surah, hit.ayah) == (hit.end_surah, hit.end_ayah) else f"{hit.ayah}-{hit.end_ayah}"
    print(f"{hit.surah}:{span:<6} {hit.name_en:<16} {hit.name_ar}   {text}")


1:5-6    Al-Fatihah       الفاتحة   إياك نعبد وإياك نستعين اهدنا الصراط المستقيم
112:1-3    Al-Ikhlas        الإخلاص   قل هو الله أحد الله الصمد لم يلد ولم يولد
2:255    Al-Baqarah       البقرة   الله لا إله إلا هو الحي القيوم لا تأخذه سنة ولا نوم
unknown   الحمد لله رب العالمين
unknown   فبأي آلاء ربكما تكذبان


## 5. Score one waveform

`score_clip` resamples to 16 kHz mono, asks for a transcript, then runs the same windowed matcher the model card describes. Whisper runs on the first call and is reused after that.


In [11]:
def to_16k_mono(samples, sample_rate: int) -> np.ndarray:
    audio = np.asarray(samples)
    if audio.ndim == 2:
        audio = audio.mean(axis=-1 if audio.shape[-1] <= 8 else 0)
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)
    if np.issubdtype(np.asarray(samples).dtype, np.integer):
        audio = audio / np.iinfo(np.asarray(samples).dtype).max
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 1.5:
        audio = audio / peak
    if sample_rate == 16000 or audio.size == 0:
        return np.ascontiguousarray(audio)
    new_len = int(round(audio.shape[0] * 16000 / sample_rate))
    if new_len < 2:
        return np.ascontiguousarray(audio[:0])
    source_x = np.linspace(0.0, 1.0, audio.shape[0], endpoint=False)
    target_x = np.linspace(0.0, 1.0, new_len, endpoint=False)
    return np.ascontiguousarray(np.interp(target_x, source_x, audio).astype(np.float32))


def report(found) -> str:
    lines = [f"length     {found.seconds:.1f}s   windows {found.windows}   agreement {found.agreement:.2f}"]
    if found.reciter is None:
        lines.append("reciter    unknown")
    else:
        voice = found.reciter.voice
        lines.append(f"reciter    {voice.name_en}  /  {voice.name_ar}")
        lines.append(f"recording  {voice.recording_name}")
        lines.append(f"score      {found.reciter.score:.3f}   ids {voice.reciter_id} / {voice.recording_id}")
    if found.verse is None:
        lines.append("verse      unknown")
    else:
        verse = found.verse
        if (verse.surah, verse.ayah) == (verse.end_surah, verse.end_ayah):
            span = str(verse.ayah)
        elif verse.surah == verse.end_surah:
            span = f"{verse.ayah}-{verse.end_ayah}"
        else:
            span = f"{verse.surah}:{verse.ayah} – {verse.end_surah}:{verse.end_ayah}"
        lines.append(f"verse      {verse.surah}:{span}  {verse.name_en}  {verse.name_ar}")
    lines.append(f"transcript {found.transcript or '(empty)'}")
    lines.append("candidates")
    if not found.matches:
        lines.append("  (none)")
    for match in found.matches:
        mark = "  *" if found.reciter is not None and match.voice.recording_id == found.reciter.voice.recording_id else "   "
        lines.append(f"{mark} {match.score:.3f}  {match.voice.name_en}  {match.voice.recording_name}")
    return "\n".join(lines)


def score_clip(samples, sample_rate: int, with_verse: bool = True) -> str:
    audio = to_16k_mono(samples, sample_rate)
    seconds = audio.shape[0] / 16000
    if seconds < MIN_SECONDS:
        return (
            f"clip is {seconds:.1f}s. 15 seconds is not enough. "
            f"Pass at least {MIN_SECONDS} seconds. {LISTEN_SECONDS} seconds is the length this model scores."
        )
    text = ""
    note = ""
    if with_verse:
        try:
            text = transcribe(audio, 16000)
        except Exception as exc:
            text = ""
            note = f"verse transcript skipped: {exc}"
    found = gallery.detect(audio, 16000, encoder_path, transcript=text)
    body = report(found)
    return f"{note}\n{body}" if note else body


## 6. Upload a file or open the microphone

Press **Identify**. Pick a file, or press the microphone icon and recite for about 30 seconds, then stop. Allow microphone access when your browser asks.

The page stays up until you interrupt the cell. A clip under 20 seconds comes back as a refusal, not a guess.


In [12]:
import gradio as gr


def identify(audio):
    if audio is None:
        return "Upload a file or record from the microphone first."
    sample_rate, data = audio
    return score_clip(data, int(sample_rate))


with gr.Blocks(title="Quran reciter detector") as demo:
    gr.Markdown(
        "Upload a recitation or record one. "
        f"Minimum {MIN_SECONDS} seconds, {LISTEN_SECONDS} seconds works best."
    )
    clip = gr.Audio(
        sources=["upload", "microphone"],
        type="numpy",
        label="Recitation",
    )
    button = gr.Button("Identify")
    result = gr.Textbox(label="Result", lines=16)
    button.click(identify, inputs=clip, outputs=result)

demo.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
